In [ ]:
!pip install kagglehub
import kagglehub

# Download latest version
path = kagglehub.dataset_download("msambare/fer2013")

print("Path to dataset files:", path)

In [ ]:
# Cell 1 – See what we got and move it to a clean folder
import os
import shutil

# This is the path kagglehub gave you (probably something like /root/.cache/kagglehub/...)
print("Original path:", path)

# List what’s inside
!ls {path}

# Usually it contains two folders: train and test
# Let's copy everything to a simple folder called fer2013
!mkdir -p /content/fer2013
!cp -r {path}/* /content/fer2013/

# Now check
!ls /content/fer2013
# You should see: train  test  (or Train  Test – case might differ)

In [ ]:
# Run this in Colab or local
!pip install torch torchvision seaborn matplotlib -q

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import transforms, datasets
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix
import numpy as np

# 1. Data loading + augmentation
data_transforms = {
    'train': transforms.Compose([
        transforms.Grayscale(num_output_channels=1), # Convert to grayscale
        transforms.RandomHorizontalFlip(),
        transforms.RandomRotation(10),
        transforms.ToTensor(),
        transforms.Normalize([0.485], [0.229])
    ]),
    'test': transforms.Compose([ # Changed 'val' to 'test'
        transforms.Grayscale(num_output_channels=1), # Convert to grayscale
        transforms.ToTensor(),
        transforms.Normalize([0.485], [0.229])
    ]),
}

# Change this path after you upload FER2013 folder
data_dir = '/content/fer2013'  # contains train/ and test/ folders. Changed 'content/fer2013' to '/content/fer2013'
image_datasets = {x: datasets.ImageFolder(f'{data_dir}/{x}', data_transforms[x])
                  for x in ['train', 'test']} # Changed 'val' to 'test'
dataloaders = {x: DataLoader(image_datasets[x], batch_size=64, shuffle=True, num_workers=2)
               for x in ['train', 'test']} # Changed 'val' to 'test'

class_names = image_datasets['train'].classes
print(class_names)
# → ['angry', 'disgust', 'fear', 'happy', 'neutral', 'sad', 'surprise']

# 2. Simple but strong model (students get 71–72% with this)
class FERModel(nn.Module):
    def __init__(self):
        super(FERModel, self).__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, 64, 3, padding=1),   nn.ReLU(), nn.BatchNorm2d(64),
            nn.Conv2d(64, 64, 3, padding=1),  nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(64, 128, 3, padding=1), nn.ReLU(), nn.BatchNorm2d(128),
            nn.Conv2d(128, 128, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(128, 256, 3, padding=1), nn.ReLU(), nn.BatchNorm2d(256),
            nn.Conv2d(256, 256, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
        )
        self.classifier = nn.Sequential(
            nn.Dropout(0.5),
            nn.Linear(256 * 6 * 6, 512),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(512, 7)
        )

    def forward(self, x):
        x = self.features(x)
        x = x.view(x.size(0), -1)
        x = self.classifier(x)
        return x

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = FERModel().to(device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)
scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=10, gamma=0.1)

# 3. Training loop
num_epochs = 30
best_acc = 0.0

for epoch in range(num_epochs):
    model.train()
    running_loss = 0.0
    for inputs, labels in dataloaders['train']:
        inputs, labels = inputs.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()

    # Validation
    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for inputs, labels in dataloaders['test']: # Changed 'val' to 'test'
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()

    val_acc = 100 * correct / total
    if val_acc > best_acc:
        best_acc = val_acc
        torch.save(model.state_dict(), 'best_fer_model.pth')

    print(f'Epoch {epoch+1}/{num_epochs} | Loss: {running_loss/len(dataloaders["train"]):.4f} | Val Acc: {val_acc:.2f}%')

print(f"Best accuracy: {best_acc:.2f}%")

In [ ]:
from google.colab import files
files.download('best_fer_model.pth')


In [ ]:
import matplotlib.pyplot as plt

epochs = list(range(1, 31))
val_acc = [40.90, 48.17, 50.57, 52.62, 56.35, 56.66, 57.61, 58.18, 57.90, 59.14,
           60.14, 60.57, 61.40, 60.23, 61.60, 61.38, 62.13, 62.64, 62.61, 62.78,
           63.22, 63.12, 62.64, 63.90, 63.69, 63.76, 63.60, 65.10, 64.04, 65.23]

plt.figure(figsize=(12, 6))
plt.plot(epochs, val_acc, marker='o', linestyle='-', color='blue', label='Validation Accuracy')
plt.title('Validation Accuracy over Epochs (FER Model)')
plt.xlabel('Epoch')
plt.ylabel('Validation Accuracy (%)')
plt.grid(True)
plt.xticks(epochs)
plt.ylim(35, 70)
plt.legend()
plt.show()
